In [1]:
import json
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, DataCollatorForLanguageModeling
from peft import LoraConfig, get_peft_model
from transformers import TrainingArguments, Trainer

C:\Users\alexy\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# -----------------------------
# 1. Cargar dataset
# -----------------------------
dataset = load_dataset("json", data_files="tutor_programacion.jsonl")
dataset = dataset["train"].train_test_split(test_size=0.1)

In [3]:
%pip install hf_xet

   ---------------------------------------- 0.0/2.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.9 MB ? eta -:--:--
   --- ------------------------------------ 0.3/2.9 MB ? eta -:--:--
   ------- -------------------------------- 0.5/2.9 MB 1.1 MB/s eta 0:00:03
   ---------- ----------------------------- 0.8/2.9 MB 1.0 MB/s eta 0:00:03
   ---------- ----------------------------- 0.8/2.9 MB 1.0 MB/s eta 0:00:03
   ---------- ----------------------------- 0.8/2.9 MB 1.0 MB/s eta 0:00:03
   ---------- ----------------------------- 0.8/2.9 MB 1.0 MB/s eta 0:00:03
   -------------- ------------------------- 1.0/2.9 MB 629.1 kB/s eta 0:00:03
   -------------- ------------------------- 1.0/2.9 MB 629.1 kB/s eta 0:00:03
   -------------- ------------------------- 1.0/2.9 MB 629.1 kB/s eta 0:00:03
   ------------------ --------------------- 1.3/2.9 MB 550.1 kB/s eta 0:00:03
   ------------------ ----------

In [4]:
# -----------------------------
# 2. Cargar modelo base
# -----------------------------
model_name = "mistralai/Mistral-7B-Instruct-v0.2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    dtype="auto"
)

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
Loading checkpoint shards: 100%|██████████| 3/3 [00:01<00:00,  1.86it/s]


In [5]:
# -----------------------------
# 3. Configurar LoRA
# -----------------------------
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM"
)

In [6]:
model = get_peft_model(model, lora_config)

In [7]:
# -----------------------------
# 4. Preprocesar el dataset
# -----------------------------
def format_instruction(example):
    prompt = (
        "Eres un tutor experto en algoritmos.\n"
        "Explica de forma clara y paso a paso.\n\n"
        f"Instrucción: {example['instruction']}\n"
        "Respuesta:\n"
    )
    return tokenizer(
        prompt + example["response"],
        truncation=True,
        padding="max_length",
        max_length=512
    )


tokenized = dataset.map(format_instruction)

Map: 100%|██████████| 15/15 [00:00<00:00, 269.17 examples/s]


In [ ]:
# -----------------------------
# 5. Entrenamiento
# -----------------------------
training_args = TrainingArguments(
    output_dir="./lora-tutor",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    logging_steps=50,
    num_train_epochs=3,
    fp16=True,
    save_steps=500,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["test"],
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False),
)


trainer.train()

The model is already on multiple devices. Skipping the move to device specified in `args`.
C:\Users\alexy\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


In [ ]:
# -----------------------------
# 6. Guardar adaptadores LoRA
# -----------------------------
model.save_pretrained("./lora-tutor")
print("Entrenamiento completado. Adaptadores guardados.")